In [ ]:
%pip install requests beautifulsoup4 lxml selenium pandas webdriver-manager

In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get('https://example.com')

html = driver.page_source   # 정적에서 response.text 와 동일
soup = BeautifulSoup(html, 'lxml')
result = soup.select_one('body > div > p:nth-child(3)')
print(result.text)

#종료
driver.quit()

Learn more


# 다나와 자동차 판매량

In [18]:
url = 'https://auto.danawa.com/auto/?Work=record'


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get(url)

html = driver.page_source   # 정적에서 response.text 와 동일
soup = BeautifulSoup(html, 'lxml')
result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div > div:nth-child(3) > div.left > table > tbody')
result = result.select('tr')
for tag in result:
    # print(tag.select_one('.num').text)
    print(tag.select_one('a').text.strip(), tag.select_one('.num').text    )

#종료
driver.quit()

기아 42,066
현대 40,066
제네시스 6,942
KGM 3,701
르노코리아 2,000


In [12]:
result.select('tr')

[<tr>
 <td class="rank">1</td>
 <td class="title">
 <a href="/auto/?Work=record&amp;Brand=307&amp;Month=2026-02-00">
 <img alt="기아" src="https://autoimg.danawa.com/photo/brand/307_40.png"/>기아
 	                            </a>
 </td>
 <td class="num">42,066</td>
 <td class="share">44.0%</td>
 </tr>,
 <tr>
 <td class="rank">2</td>
 <td class="title">
 <a href="/auto/?Work=record&amp;Brand=303&amp;Month=2026-02-00">
 <img alt="현대" src="https://autoimg.danawa.com/photo/brand/303_40.png"/>현대
 	                            </a>
 </td>
 <td class="num">40,066</td>
 <td class="share">41.9%</td>
 </tr>,
 <tr>
 <td class="rank">3</td>
 <td class="title">
 <a href="/auto/?Work=record&amp;Brand=304&amp;Month=2026-02-00">
 <img alt="제네시스" src="https://autoimg.danawa.com/photo/brand/304_40.png"/>제네시스
 	                            </a>
 </td>
 <td class="num">6,942</td>
 <td class="share">7.3%</td>
 </tr>,
 <tr>
 <td class="rank">4</td>
 <td class="title">
 <a href="/auto/?Work=record&amp;Brand=326&a

# 다나와 2025년 1월부터 12월까지 국산 자동차 판매대수 수집

https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month=2025-01-00

In [54]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

company, amount = [],[]
year,month = [],[]


# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)


for _month in range(1,13):
    _year = '2025'
    _url = f'https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month={_year}-{_month:02}-00'   
    

    # 페이지 열기
    driver.get(_url)
    time.sleep(1)  # 페이지 완전 로딩될때까지 기다림.. 대략 1초면 충분하다고 판단

    html = driver.page_source   # 정적에서 response.text 와 동일
    soup = BeautifulSoup(html, 'lxml')
    result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div:nth-child(1) > table > tbody')
    result = result.select('tr')
    count = len(result)
    for tag in result:        
        # print(tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] )
        co, am =  tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] 
        company.append(co); amount.append(am)
    year += [_year]*count
    month += [_month]*count
        

#종료
driver.quit()

In [89]:
import pandas as pd
data = {
    'year' : year,
    'month' : month,
    'company' : company,
    'amount' : amount
}
df = pd.DataFrame(data)
df['amount'] = df['amount'].str.replace(',','').astype(int)
df.head()

,year,month,company,amount
0,2025,1,기아,38412
1,2025,1,현대,37230
2,2025,1,제네시스,8824
3,2025,1,르노코리아,2601
4,2025,1,KGM,2300


# mysql 데이터베이스 적재 하기

In [90]:
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
import os
# 환경변수 읽어오기
load_dotenv()

# 환경변수에서 MySQL 연결 정보 로드
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'car_db'),
    'port': int(os.getenv('DB_PORT', 3306))
}

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
query = '''
insert into sale_car(year,month,company,amount) 
values(%s,%s,%s,%s)
'''

for _, row in df.iterrows():
    params = (row.year, row.month,row.company, row.amount)
    cursor.execute(query,params=params)

conn.commit()

cursor.close()
conn.close()
print('데이터 삽입 완료')

데이터 삽입 완료


In [ ]:
for _, row in df.iterrows():
    print(row.year, row.month,row.company, row.amount)